In [1]:
%%capture
!pip install ogb scikit-learn

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import torch
from ogb.linkproppred import LinkPropPredDataset
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

DRIVE_OUT = '/content/drive/MyDrive/DDI_project/coverage_analysis/'
print('Ready.')

Mounted at /content/drive
Ready.


In [4]:
def to_numpy(x):
    if hasattr(x, 'numpy'):
        return x.numpy()
    return np.array(x)

train_pos = to_numpy(split_edge['train']['edge'])
val_pos   = to_numpy(split_edge['valid']['edge'])
val_neg   = to_numpy(split_edge['valid']['edge_neg'])
test_pos  = to_numpy(split_edge['test']['edge'])
test_neg  = to_numpy(split_edge['test']['edge_neg'])

print(f'Train positives: {len(train_pos):,}')
print(f'Val positives:   {len(val_pos):,}')
print(f'Test positives:  {len(test_pos):,}')

Train positives: 1,067,911
Val positives:   133,489
Test positives:  133,489


In [5]:
# Load all embeddings
pubmedbert = np.load(DRIVE_OUT + 'pubmedbert_embeddings.npy')
pubmedbert_ids = np.load(DRIVE_OUT + 'pubmedbert_drug_ids.npy')

chemberta = np.load(DRIVE_OUT + 'chemberta_embeddings.npy')
chemberta_ids = np.load(DRIVE_OUT + 'chemberta_drug_ids.npy')

# Load OGB-ddi mapping to align node idx → embedding row
mapping_df = pd.read_csv(
    '/content/ogb_data/ogbl_ddi/mapping/nodeidx2drugid.csv.gz',
    compression='gzip', header=0)
mapping_df.columns = ['node_idx', 'drugbank_id']

# Build node_idx → embedding index lookup
def build_node_emb_matrix(embeddings, drug_ids, mapping_df):
    id_to_row = {did: i for i, did in enumerate(drug_ids)}
    n_nodes = len(mapping_df)
    dim = embeddings.shape[1]
    mean_vec = embeddings.mean(axis=0)
    matrix = np.zeros((n_nodes, dim))
    for _, row in mapping_df.iterrows():
        idx = int(row['node_idx'])
        did = row['drugbank_id']
        matrix[idx] = id_to_row.get(did, -1) != -1 and embeddings[id_to_row[did]] or mean_vec
    return matrix

# Simpler and faster version
def build_matrix(embeddings, drug_ids, mapping_df):
    id_to_emb = dict(zip(drug_ids, embeddings))
    mean_vec = embeddings.mean(axis=0)
    matrix = np.array([
        id_to_emb.get(row['drugbank_id'], mean_vec)
        for _, row in mapping_df.iterrows()
    ])
    return matrix

pubmedbert_matrix = build_matrix(pubmedbert, pubmedbert_ids, mapping_df)
chemberta_matrix  = build_matrix(chemberta,  chemberta_ids,  mapping_df)

print(f'PubMedBERT matrix: {pubmedbert_matrix.shape}')
print(f'ChemBERTa matrix:  {chemberta_matrix.shape}')

PubMedBERT matrix: (4267, 768)
ChemBERTa matrix:  (4267, 768)


In [6]:
def make_pair_features(edges, emb_matrix):
    src = edges[:, 0]
    dst = edges[:, 1]
    return np.concatenate([emb_matrix[src], emb_matrix[dst]], axis=1)

def subsample(edges, n=50000, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(edges), size=min(n, len(edges)), replace=False)
    return edges[idx]

In [7]:
def evaluate(clf, X_test_pos, X_test_neg):
    X_test = np.vstack([X_test_pos, X_test_neg])
    y_test = np.array([1]*len(X_test_pos) + [0]*len(X_test_neg))
    scores = clf.predict_proba(X_test)[:, 1]
    auroc = roc_auc_score(y_test, scores)
    aupr  = average_precision_score(y_test, scores)
    return auroc, aupr

In [8]:
embeddings_to_test = {
    'PubMedBERT': pubmedbert_matrix,
    'ChemBERTa':  chemberta_matrix,
}

classifiers = {
    'LR':  LogisticRegression(max_iter=1000, C=1.0),
    'RF':  RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=200, random_state=42),
}

results = []

for emb_name, emb_matrix in embeddings_to_test.items():
    print(f'\n=== {emb_name} ===')

    # Build training data (subsampled)
    train_pos_sub = subsample(train_pos, n=50000)
    train_neg_sub = subsample(
        np.array([[i,j] for i,j in zip(
            np.random.randint(0, 4267, 50000),
            np.random.randint(0, 4267, 50000)
        )]), n=50000)

    X_train = np.vstack([
        make_pair_features(train_pos_sub, emb_matrix),
        make_pair_features(train_neg_sub, emb_matrix)
    ])
    y_train = np.array([1]*len(train_pos_sub) + [0]*len(train_neg_sub))

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    # Build test data
    X_test_pos = scaler.transform(make_pair_features(test_pos, emb_matrix))
    X_test_neg = scaler.transform(make_pair_features(test_neg, emb_matrix))

    for clf_name, clf in classifiers.items():
        print(f'  Training {clf_name}...', end=' ')
        clf.fit(X_train_scaled, y_train)
        auroc, aupr = evaluate(clf, X_test_pos, X_test_neg)
        print(f'AUROC={auroc:.4f}  AUPR={aupr:.4f}')
        results.append({
            'embedding': emb_name,
            'model': clf_name,
            'auroc': round(auroc, 4),
            'aupr': round(aupr, 4)
        })

results_df = pd.DataFrame(results)
print('\n=== Summary ===')
print(results_df.to_string(index=False))
results_df.to_csv(DRIVE_OUT + 'classical_baselines_results.csv', index=False)
print('\nSaved.')


=== PubMedBERT ===
  Training LR... AUROC=0.7192  AUPR=0.7694
  Training RF... AUROC=0.7852  AUPR=0.7987
  Training MLP... AUROC=0.7554  AUPR=0.7951

=== ChemBERTa ===
  Training LR... AUROC=0.6796  AUPR=0.7325
  Training RF... AUROC=0.7772  AUPR=0.8059
  Training MLP... AUROC=0.7380  AUPR=0.7809

=== Summary ===
 embedding model  auroc   aupr
PubMedBERT    LR 0.7192 0.7694
PubMedBERT    RF 0.7852 0.7987
PubMedBERT   MLP 0.7554 0.7951
 ChemBERTa    LR 0.6796 0.7325
 ChemBERTa    RF 0.7772 0.8059
 ChemBERTa   MLP 0.7380 0.7809

Saved.


In [ ]:
print('\n=== Ensemble (PubMedBERT + ChemBERTa concatenated) ===')

# Build combined training data
train_pos_sub = subsample(train_pos, n=50000, seed=42)
rng = np.random.default_rng(42)
train_neg_idx = np.column_stack([
    rng.integers(0, 4267, 50000),
    rng.integers(0, 4267, 50000)
])

# Combined features = PubMedBERT concat ChemBERTa per drug
# Each pair: [emb_A_pm | emb_B_pm | emb_A_ch | emb_B_ch] → 3072-dim
def make_combined_features(edges, emb_pm, emb_ch):
    src, dst = edges[:, 0], edges[:, 1]
    return np.concatenate([
        emb_pm[src], emb_pm[dst],
        emb_ch[src], emb_ch[dst]
    ], axis=1)

X_train = np.vstack([
    make_combined_features(train_pos_sub, pubmedbert_matrix, chemberta_matrix),
    make_combined_features(train_neg_idx, pubmedbert_matrix, chemberta_matrix)
])
y_train = np.array([1]*50000 + [0]*50000)

scaler_combined = StandardScaler()
X_train_scaled = scaler_combined.fit_transform(X_train)

X_test_pos = scaler_combined.transform(
    make_combined_features(test_pos, pubmedbert_matrix, chemberta_matrix))
X_test_neg = scaler_combined.transform(
    make_combined_features(test_neg, pubmedbert_matrix, chemberta_matrix))

ensemble_clfs = {
    'LR':  LogisticRegression(max_iter=1000, C=1.0),
    'RF':  RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=200, random_state=42),
}

for clf_name, clf in ensemble_clfs.items():
    print(f'  Training {clf_name}...', end=' ')
    clf.fit(X_train_scaled, y_train)
    auroc, aupr = evaluate(clf, X_test_pos, X_test_neg)
    print(f'AUROC={auroc:.4f}  AUPR={aupr:.4f}')
    results.append({
        'embedding': 'PubMedBERT+ChemBERTa',
        'model': clf_name,
        'auroc': round(auroc, 4),
        'aupr': round(aupr, 4)
    })

# Updated summary
results_df = pd.DataFrame(results)
print('\n=== Full Summary ===')
print(results_df.to_string(index=False))
results_df.to_csv(DRIVE_OUT + 'classical_baselines_results.csv', index=False)
print('\nSaved.')


=== Ensemble (PubMedBERT + ChemBERTa concatenated) ===
  Training LR... 